# FENCINGLY · 01 · Extracción a duración completa y cadencia nativa

**Qué hace:** extrae poses de **todos** los fotogramas de cada clip, sin recortar la ventana y sin
remuestrear, y **va subiendo por lotes** a Hugging Face conforme avanza.

**Si Colab se corta, no se pierde el trabajo.** Al volver a ejecutar, el cuaderno lee lo que ya hay
—en disco y **en el repositorio remoto**— y sigue por donde se quedó. Un corte cuesta como mucho
**un lote**, no la corrida.

**El estado no es un contador.** Un clip cuenta como hecho si su `.npz` existe con fila en el
índice, o si su lote ya está subido. Durante el ensayo un contador dijo «completado = 3» habiendo 6
ficheros, y ese es el fallo que este diseño evita. El fichero `estado_corrida.json` es un
**registro de auditoría**, no la autoridad: si lo borras, la reanudación funciona igual.

**El cuaderno no contiene lógica.** Llama a `pasos.py` (los pasos), `corrida.py` (los lotes),
`respaldo.py` (Hugging Face) y `barra.py` (las barras). Todo eso se ejecutó completo en local antes
de subirlo, así que lo que corre aquí es lo que se midió allí.

## Arranque · dependencias, código y token

**El token va en un secreto de Colab, no en una celda.** Icono de la **llave** en la barra izquierda
→ *Add new secret* → nombre `HUGGINFACE`, valor tu token de **escritura**, y activa
*Notebook access*. Se pone una vez y no vuelve a pedirse ni al reiniciar el runtime.

Escribir en el Hub **exige token aunque el repositorio sea público**: lo público quita la
autenticación para *leer*, no para *escribir*. Y un token de **solo lectura** autenticaría sin
fallo aquí y reventalía al subir el primer lote, o sea horas después: por eso esta celda
comprueba el permiso **antes** de gastar GPU.

In [ ]:
import subprocess, sys
for paquete in ("huggingface_hub",):
    try:
        __import__(paquete)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

from pathlib import Path
for p in ("/content/FENCINGLY/catalogo", "/content", str(Path.cwd())):
    if p not in sys.path:
        sys.path.insert(0, p)

import respaldo as R, corrida as CO, barra as B

# --- los dos repositorios, cada uno con su privacidad ESPERADA ---
REPO_DATOS  = "EXCALORIO/fencingly-datos"        # PUBLICO: poses, tablas, indices (seudonimos)
REPO_CODIGO = "EXCALORIO/fencingly-extraccion"   # PRIVADO: recortes, nombres, cuadernos

SUBIR_A = REPO_DATOS      # a donde van los lotes de esta corrida
ES_PUBLICO = True         # si NO cuadra con el remoto, la celda levanta y no sube nada

print("motor de barras:", B.motor())
SES = R.sesion(exigir=True)
API = SES["api"]
R.verifica_repo(API, SUBIR_A, privado_esperado=not ES_PUBLICO)

## Paso 0 · Puertas

**Levantan, no avisan.** Versiones exactas de producción, huella del productor y parche de
ventana.

**Dos raíces, y no son la misma.** `RAIZ_CODIGO` es el árbol con el que se **extrae** (puede
ser una copia congelada). `RAIZ_MEDIDA` es el árbol **vivo**, el único con el que la huella
significa algo. Pedirle la huella a la copia congelada da `fd2421b3…` en vez de `3e1dbe0a…`
**sin ningún error** — es el punto 19 de `AGENTS.md`, y esta puerta lo cazó la primera vez que
se ejecutó porque los dos argumentos están separados.

**`tqdm` no se da por hecho.** No está instalado en `.venv` ni en `.venv-gpu`, y
`pipeline.process_many` hace `try: from tqdm.auto import tqdm … except: bar = None`, así que
**en local la barra no existe y no avisa**. `barra.py` usa tqdm si está y un respaldo propio
con el mismo API si no.

In [ ]:
import os, sys, json
from pathlib import Path

# --- rutas: ajustar solo este bloque ---
BASE        = Path("/content/FENCINGLY")          # raiz del paquete en Colab
RAIZ_CODIGO = BASE / "codigo_congelado"           # con el que se EXTRAE
RAIZ_MEDIDA = BASE / "webapp"                     # arbol VIVO: la huella sale de aqui
WS          = BASE / "ws_completo"                # workspace de salida
DIR_MP4     = BASE / "mp4"                        # o el Drive montado
MODELO_INDEX = BASE / "modelo_index.csv"          # un index.csv de 119 columnas QUE YA CARGA
PESOS       = BASE / "yolo11m-pose.pt"
MANIFIESTO  = BASE / "manifest_foil.csv"
EXTRA       = []          # ficheros JSON/CSV o URLs de JSON con clips nuevos

for p in (str(RAIZ_CODIGO), str(BASE / "catalogo")):
    if p not in sys.path:
        sys.path.insert(0, p)

import pasos
ENT = pasos.paso0_puertas(RAIZ_CODIGO, raiz_medida=RAIZ_MEDIDA, estricto=True)

## Paso 1 · Manifiesto, y cómo añadirle clips nuevos

**Extensión:** `EXTRA` acepta ficheros `.json` / `.jsonl` / `.csv` **o una URL de JSON**. Cada
fuente se valida contra el esquema mínimo (`clip_id`); la que no lo cumple **se rechaza con su
motivo** y no aborta la corrida. En un `clip_id` duplicado se queda **la primera**, igual que
`AnalysisRepository` con el orden de `--npz-dir`.

**Filtro de florete, explícito.** `weapon == "foil"`, comprobado en la tabla. Los clips **sin
arma se rechazan**, no se asumen florete — el fichero se llame `manifest_foil.csv` o como sea.
Si el manifiesto no trae columna `weapon`, la celda **levanta**.

In [ ]:
MAN = pasos.paso1_manifiesto(MANIFIESTO, extra=EXTRA, arma="foil")
MAN.head()

## Paso 1b · Clips nuevos: del CDN y de un JSON por URL

Los clips están en **`clips.fencingdatabase.com/<torneo>/<clip_id>.mp4`**. Las páginas HTML del
sitio devuelven **403** a un cliente que no sea navegador, y eso es lo que hacía que pareciera
inaccesible; el CDN no lo hace.

`lee_manifiesto` acepta CSV, JSON local **o una URL de JSON**, y **filtra por arma siempre**. Una
fila **sin** `weapon` se **descarta**, no se asume florete: asumirla metería espada en el corpus sin
ningún aviso.

Medido en `hongkongfoil2026`: **1920×1080 a 60 fps**. El material nuevo gana **cadencia** (pasos de
16,7 ms) pero **no resolución** — sigue siendo 1080p como Turín 2023 y Tokio 2020.

In [ ]:
import fuentes as FU

# Deja aqui los manifiestos nuevos: rutas locales o URLs de JSON. Vacio = no anade nada.
MANIFIESTOS_NUEVOS = []

NUEVOS = []
for origen in MANIFIESTOS_NUEVOS:
    NUEVOS += FU.lee_manifiesto(origen, arma="foil")
print("clips nuevos de florete:", len(NUEVOS))

if NUEVOS:
    pares = [(r["tournament_id"], r["clip_id"]) for r in NUEVOS if r["tournament_id"]]
    BAJADOS = FU.descarga_lote(pares, DIR_MP4, barra=B)
    ok = [r for r in BAJADOS if r["ruta"]]
    print("descargados:", len(ok), "de", len(BAJADOS))
    if ok:
        m = FU.mide(ok[0]["ruta"])
        print("primero:", m["ancho"], "x", m["alto"], "@", m["fps"], "fps")
        print("  el fuerte medira", m["ancho_fuerte_px"], "px de ancho y la cazoleta",
              m["diam_cazoleta_px"], "px")

## Paso 2 · Estado: qué está hecho, cruzando disco **y** remoto

Aquí está la diferencia con una reanudación ingenua. En un runtime **nuevo** `/content` está
vacío, así que mirar solo el disco haría reextraer todo. Esta celda descuenta además **los
lotes que ya están subidos**, que es lo que hace que reanudar en otra sesión no repita trabajo.

In [ ]:
SEDES = CO.sedes_del_manifiesto(MAN)
PEND = CO.pendientes(MAN, WS, pasos, api=API, repo=SUBIR_A, sedes_de=SEDES)
print("\na extraer en esta corrida:", len(PEND["pendiente"]))

CO.informe(WS)     # la historia de lo que ya se hizo, si hubo una corrida antes

## Paso 3 · Extracción **por lotes**, subiendo conforme avanza

**Por qué por lotes.** En Colab los ficheros viven en `/content`, que **muere con el runtime**.
Extraer 6.000 clips y subirlos al final significa que un corte en el 5.999 tira los 5.999.

`TAM_LOTE = 200` no es arbitrario: a los **17,2 s/clip** medidos son unos **57 minutos** de trabajo
por lote, o sea lo máximo que puede costar un corte. Y son ~12 MB de poses por subida, que no
satura el remoto. Bajarlo multiplica subidas sin ganar nada; subirlo arriesga más trabajo.

Verás **dos barras con ETA**: una de lotes y, dentro, la de la extracción clip a clip.

`borrar_tras_subir=False` a propósito: perder el `.npz` local por una subida que falló en
silencio sería peor que llenar el disco.

In [ ]:
from fencingly_v2 import pipeline as PL, pose as P, video as V
from fencingly_v2 import colab_workflow as CW

extractor = CW.create_extractor(str(PESOS), pasos.PLAN_POSE)
extractor.batch = pasos.PLAN_POSE["pose_batch"]
print("modelo cargado: imgsz", extractor.imgsz, "lote", extractor.batch)

# clip_id -> ruta del mp4. El nombre del fichero ES el clip_id.
RUTAS = {p.stem: str(p) for p in Path(DIR_MP4).rglob("*.mp4")}
print("mp4 localizados:", len(RUTAS))

COR = CO.por_lotes(PEND["pendiente"], RUTAS, WS, extractor, pasos, PL, P, V,
                   modelo_index=MODELO_INDEX, man=MAN,
                   api=API, repo=SUBIR_A, sedes_de=SEDES,
                   tam_lote=CO.TAM_LOTE, subir=True, borrar_tras_subir=False)
RES = None   # el indice ya se escribio dentro de cada lote

## Paso 4 · Índice, contado por el lector real

Se escribe con el **orden de columnas de un índice que ya carga** (119), y se emite fila
**solo si su `.npz` está en disco**. Después se cuenta con `AnalysisRepository.index()`, **no
con las filas del CSV**: eso es lo que falló anoche con 689 clips.

In [ ]:
import pandas as pd
IDX = pasos.paso4_indice(RES, WS, MODELO_INDEX, manifiesto=MAN)

# En una RE-EJECUCION sin pendientes, `paso4_indice` no reescribe el indice (y hace bien:
# no lo sobreescribe con nada). Entonces `IDX` viene vacio y verificar contra el daria un
# OK que no comprueba nada. Asi que lo esperado se toma del indice EN DISCO.
if len(IDX):
    esperados = set(IDX["clip_id"].astype(str))
else:
    f = WS / "npz" / "index.csv"
    esperados = (set(pd.read_csv(f, usecols=["clip_id"], low_memory=False)["clip_id"].astype(str))
                 if f.is_file() else set())
    print(f"  re-ejecucion: se verifica contra el index.csv en disco ({len(esperados)} clips)")

VER = pasos.paso4b_verifica_lector(WS, RAIZ_MEDIDA, esperados)
assert esperados, "no hay nada que verificar: ni index.csv nuevo ni en disco"
assert VER.get("ok") is not False, "hay clips INVISIBLES para el lector: revisar antes de seguir"

## Paso 4c · ¿Se cuela alguien de otra pista?

El proyecto ya evita al árbitro y al público (banda `(0,45, 0,97)`, blancura del torso porque el
uniforme **es blanco**, y penalización por quietud). Lo que no había era **una cifra**.

Medido sobre **10.618 clips de 21 sedes**: **88,36 % limpios** por los cuatro detectores. Y dos cosas
que esa medida descubrió:

- **La banda no es una puerta dura, es una penalización.** Si lo fuera, el **1,99 %** con los pies
  fuera en más de la mitad de los cuadros tendría que ser cero.
- **El desnivel de pies no prueba contaminación por sí solo**: la perspectiva legítima lo produce,
  porque un tirador en el extremo lejano de la misma pista tiene los pies más arriba. Ese 10,52 % es
  una **cota superior de sospecha**.

El detector que sí es preciso es **`razon_alturas`** (0,99 %): a la misma profundidad dos tiradores
no difieren un 60 % de tamaño. Inspeccionados sus dos peores casos, los dos eran contaminación
inequívoca — en uno las cajas estaban sobre **un espectador de la grada** y sobre **suelo vacío**
mientras el tirador real quedaba sin marcar.

**Esto no borra clips.** Da columnas y una bandera; descartar el 11,64 % por una cota superior sería
peor que la contaminación.

In [ ]:
import aislar as AI
import numpy as np

AUD = []
for f in sorted((WS / "npz").glob("*.npz")):
    try:
        with np.load(f, allow_pickle=True) as z:
            if "poses" not in z.files or "boxes" not in z.files:
                continue
            r = AI.audita(z["poses"], z["boxes"])
    except Exception:
        continue
    r["clip_id"] = f.stem
    AUD.append(r)

RES_AIS = AI.resumen(AUD)

# Los peores, con su motivo nombrado y su fuerza declarada
import pandas as pd
da = pd.DataFrame(AUD)
if len(da) and "razon_alturas" in da.columns:
    for _, r in da.nlargest(5, "razon_alturas").iterrows():
        print("  %-40s %s" % (r["clip_id"][:40], AI.motivo(r.to_dict())))

## Paso 5 · Rodaja de entrenamiento — **mando aparte, y no toca los NPZ**

**`store.T_MAX = 135` y el almacén alinea por la DERECHA.** Un clip de 500 fotogramas a 50 fps
se queda con los últimos 135 = **2,70 s, el 27 % del clip**, y tira el resto **sin dar error**.
Y `dataset.prepare` rehace el eje a 30 fps, que en los clips rápidos borra el **44,4 %**.
**Ninguna de las dos cosas avisa.**

Por eso esto es un paso separado y con `aplicar=False`: **imprime lo que se perdería** con el
anclaje y el `t_max` que elijas, y no modifica nada. Anclajes: `"tocado"` (para el almacén y
las medidas) o `"allez"` (para la escala).

In [ ]:
ROD = pasos.paso5_rodaja(WS, t_max=135, anclaje="tocado", pre_s=4.0, post_s=0.3, aplicar=False)
# y con el otro ancla, para comparar:
_ = pasos.paso5_rodaja(WS, t_max=600, anclaje="allez", aplicar=False)

## Paso 6 · Tabla por fotograma (Parquet)

Una fila por `(clip_id, frame_idx)`. **`NaN` significa indefinido, nunca cero**: un cero es una
posición, un `NaN` es «no se midió».

**`src_fps` y el ratio `fps/src_fps` viajan los dos como columnas, y son lo más importante de
esta tabla.** `stage_save` calcula `src_fps = fps · src_ratio` cuando `dup_mode == "regular"`:
la cadencia **real** de la fuente. Sin ellos, «50 fps» y «60 fps con origen a 15 fps» se tratan
igual **y no lo son**.

**Y hay un artefacto mayoritario que no estaba en ninguna lista.** Medido sobre los 15.452
clips del índice vivo (`dup_mode` presente en el 100 %), por fps **nominal redondeado**:

| nominal | clips | patrón regular | ratio ≥ 2 |
|---|---:|---:|---:|
| 25 | 2.894 | 10 (0,3 %) | **0** |
| **30** | **11.211** | **8.503 (75,8 %)** | 35 |
| **50** | 649 | **0 (0,0 %)** | **0** |
| 60 | 170 | 66 (38,8 %) | **19 (11,2 %)** |

- **El problema es la banda de 30, que es la mayoritaria:** 8.503 clips con **telecine 25→30**.
  `dup_rate` mediana **0,164**, `dup_period` **6,0 fotogramas exactos**, `src_fps` mediana
  **25,0**. **Uno de cada seis fotogramas es un duplicado**, y un duplicado da **velocidad cero
  en un paso y doble en el siguiente**.
- **La cámara lenta es residual: 54 clips de 15.452, el 0,35 %.** Ratios 2,0 · 2,4 · 4,0 · 6,0
  · **12,0**. Un ratio de 12 son **once fotogramas de velocidad cero seguidos de uno con
  velocidad doce veces mayor** — *eso* es lo que rompe una derivada, no la cadencia nominal.
  Sedes: eurochamps2022 (19), tbilisifoil2023 (19), worldchamps2022 (8), serbiafoil2022 (4).
- **A 50 fps: cero clips con patrón regular.** Es lo que sostiene el argumento del paso de
  20 ms. A 60 fps el 38,8 % tiene duplicados pero solo **19 de 170** están multiplicados de
  verdad; el resto es conversión 50→60 a ratio 1,20, material real reempaquetado, **no lento**.

Tres consecuencias, y son obligatorias:

1. **`src_fps` y el ratio viajan siempre.**
2. **El suavizado va ANTES de derivar**, y su ventana se justifica **contra un duplicado cada
   seis fotogramas** —no contra la cámara lenta—: por debajo de ~7 fotogramas no absorbe el
   artefacto. Guardar **cruda y suavizada**, las dos.
3. **Toda medida de velocidad se reporta con `src_fps` al lado**, no solo con la nominal.

> **Aviso de denominadores.** El censo completo tiene **29.220 clips con fps** (20.938 a 30,
> 4.883 a 25, 1.547 a 50, 816 a 60), pero `dup_mode` está medido en **15.452 (52,9 % del censo,
> 100 % del índice vivo)**. **No multiplicar la tasa por el censo:** la muestra medida no es
> aleatoria, es la que se llegó a procesar, y el residuo está sesgado (la tasa de detección
> correlaciona **r = −0,861** con la fracción ya extraída). Citar las dos cifras con su n.

`escala_r` va **cruda y continua**, con su `NaN`. No es una bandera: ver el paso 7.

In [ ]:
TAB = pasos.paso6_tabla(WS, salida_parquet=WS / "metricas_por_fotograma.parquet")
TAB.head()

## Paso 6b · Señales extra: zumbador, cazoleta y marcador

Tres cosas que se guardan **como columnas con su cobertura**, no como veredictos.

**El zumbador ya estaba implementado** en `fencingly_v2.audio_touch`, con búsqueda adaptativa,
porque **la frecuencia depende del aparato**. Medido en 7 sedes: mediana **3.188 Hz**, rango
**2.929–4.204** (Serbia 3.102, Incheon 3.186, Hong Kong 2026 3.226, Turín 3.252, Euros 3.773,
Mundial 3.998). Una banda de 2.000–2.700 Hz **no captura ni uno de los 15 tonos detectados**, así
que no toques la banda sin medir. Y **solo los clips de 2022 en adelante traen audio**.

**La cazoleta va como proponente, no como medida.** Con `the hilt of a sword` sobre recorte ampliado
8×: coloca a menos de un diámetro en **0,33** de 120 recortes, contra **0,04** del señuelo verbal
`a banana` y **0,61** del control `a hand`. El margen de **+0,29** es lo que dice que la señal es del
prompt y no del prior. **La confianza no sirve para filtrar**: los aciertos promedian 0,238 y los
fallos 0,253.

Y **`a bell guard`, el término técnico correcto, devuelve exactamente cero** en todas las sedes.

In [ ]:
# --- zumbador: el modulo del proyecto, no una banda nueva ---
from fencingly_v2 import audio_touch as AT

ZUM = []
for cid in list(EST.get("hecho", []))[:200] if isinstance(EST, dict) else []:
    ruta = RUTAS.get(cid)
    if not ruta:
        continue
    try:
        z = AT.buzzer_onset(ruta)
        ZUM.append({"clip_id": cid, "t_buzzer_s": z.get("t_buzzer_s"),
                    "f_peak_hz": z.get("f_peak_hz"), "snr_db": z.get("snr_db"),
                    "conf": z.get("confidence")})
    except Exception as exc:
        ZUM.append({"clip_id": cid, "t_buzzer_s": None, "f_peak_hz": None,
                    "snr_db": None, "conf": 0.0, "motivo": type(exc).__name__})
if ZUM:
    import pandas as pd
    dz = pd.DataFrame(ZUM)
    con = dz[dz.f_peak_hz.notna()]
    print("zumbador: tono en %d de %d clips" % (len(con), len(dz)))
    if len(con):
        print("  frecuencia mediana %.0f Hz   rango %.0f-%.0f" % (
            con.f_peak_hz.median(), con.f_peak_hz.min(), con.f_peak_hz.max()))
        print("  AVISO: los clips anteriores a 2022 no traen audio; ahi el fallo es esperado")

# --- cazoleta: proponente. Necesita acceso concedido a facebook/sam3 ---
HACER_CAZOLETA = False   # ponlo a True cuando tengas los pesos de SAM3
if HACER_CAZOLETA:
    import cazoleta as CZ
    proc_s3, mod_s3 = CZ.carga_sam3(PESOS_SAM3)
    print("proponente de cazoleta listo. Recuerda: 0,33 de colocacion, precision SIN verificar.")

## Paso 7 · Huecos declarados

**La mitad del valor de este cuaderno.** Si algo no se puede sacar de vídeo de retransmisión a
25-60 fps con lo que hay, saberlo ahorra el intento.

In [ ]:
pasos.paso7_huecos()

## Traspaso al cuaderno 02

Lo que el 02 necesita es el **workspace** y su `npz/index.csv`. Esta celda sube el índice y la
tabla, y deja escrito dónde está todo para que el 02 no tenga que adivinarlo.

`QUITAR_NOMBRES` sube el índice **sin** las columnas `fotl_name`, `fotr_name` y `bout_id`, que son
las **tres de 119** que contienen nombres de personas (medido sobre 16.933 filas escaneando las 119,
no adivinando por el nombre de columna). Está en `False` porque los resultados de competición son
registro público. El mando se queda por si alguna vez hace falta lo contrario: es una línea.

In [ ]:
import pandas as pd

QUITAR_NOMBRES = False
CON_NOMBRE = ("fotl_name", "fotr_name", "bout_id")   # las 3 de 119, medido

f = WS / "npz" / "index.csv"
if f.is_file():
    idx = pd.read_csv(f, low_memory=False)
    if QUITAR_NOMBRES:
        fuera = [c for c in CON_NOMBRE if c in idx.columns]
        salida = idx.drop(columns=fuera)
        assert not [c for c in CON_NOMBRE if c in salida.columns], "quedan columnas con nombres"
        print("columnas quitadas:", fuera or "ninguna presente")
    else:
        salida = idx
    print("columnas subidas:", len(salida.columns), "de", len(idx.columns))
    destino = WS / "index_subida.csv"
    salida.to_csv(destino, index=False)
    R.respalda_fichero(API, SUBIR_A, destino, "data/index/index.csv", "indice de la corrida")

tp = WS / "metricas_por_fotograma.parquet"
if tp.exists():
    R.respalda_fichero(API, SUBIR_A, tp, "data/tables/metricas.parquet", "tabla por fotograma")

print()
print("PARA EL CUADERNO 02:")
print("  FENCINGLY_WORKSPACE =", WS)
print("  indice              =", f)
print("  repo de datos       =", SUBIR_A)
CO.informe(WS)

---

## Riesgo abierto número uno: contaminación silenciosa por planos de repetición

`PeriodicSlipperyChipmunk` (50 fps, `tokyoolympics2020`) **abre con primeros planos de
repetición del tocado**, con un barrido de anillos olímpicos, y solo después pasa al plano en
vivo. La ventana de 4,3 s se quedaba con el plano en vivo; la duración completa mete la
repetición dentro.

**Y la cobertura de pose es 1,000 en L y en R.** El detector encuentra a los dos tiradores
porque **están ahí**, en otro encuadre y a otra escala, y **ninguna comprobación existente lo
marca**. Que la cobertura bajara era una alarma; aquí no hay ninguna. `cuts` reporta **1**
donde a la vista hay tres planos.

Las cuatro vías probadas y por qué ninguna caza:

| vía | resultado |
|---|---|
| `dup_rate` | ciego a 50 fps: mediana 0,000, cero `dup_mode="regular"` |
| `cuts` | sub-detecta: 1 contra ≥3 planos visibles |
| `reloj_parado_todo_el_clip` | >30,5 fps **20,0 %** [14,2, 27,4] vs ≤30,5 fps **20,5 %** [11,2, 34,5]. Solapan, −0,5 pp: no separa |
| escala aparente de caja | control `r_max = 1,58` contra umbral 1,60; clip limpio 1,32. **1,32 y 1,58 se solapan** |

**No se bajó el umbral a 1,55 para que el control pasara.** Un umbral elegido para que pase el
control es un umbral que mide el control — es el error del umbral heredado del reloj, al revés.

Y el primer intento es más instructivo que el segundo: normalizar por la mediana **del clip
entero**, cuando la repetición es la mitad del clip, **contamina el normalizador con lo que se
quiere detectar**. Es el mismo error de clase que el nulo circular preservando las columnas de
nivel.

## Confusor de sede en la cadencia

Los 904 clips de alta cadencia salen de **tres sedes**: `tokyoolympics2020` (637),
`tokyofoil2022` (190), `tbilisifoil2023` (77). Y los **154** clips de 50 fps con corte de plano
son **todos** de `tokyoolympics2020`, los 154.

**O sea: «alta cadencia» es en buena parte «Tokio 2020».** Cualquier comparación por cadencia
arrastra el confusor de sede, y alguien lo va a leer como un efecto de la cadencia. Es la
tercera vez en la sesión que un confusor de sede muerde.

## Y una máscara que no basta

Tapar `y > 0,82·H` **no es suficiente**: el marcador del árbitro con sus lámparas puede quedar
**por encima** de esa línea. Este cuaderno no recorta nada, pero cualquier consumidor de estos
NPZ que aplique esa máscara tiene que saberlo.